In [0]:
# --- CORRECTED: SALES DATA CUBE ---
# Aggregates strictly on the Sales grain. No inventory joins to prevent data duplication.

spark.sql("""
CREATE OR REPLACE TABLE de_mini_project.gold.sales_cube AS
SELECT 
    date_format(s.transaction_date, 'yyyy-MM') AS sale_month,
    s.product_id,
    s.store_id,
    
    -- KPI 1, 3, 7: Net Sales (Excluding Returns)
    ROUND(SUM(CASE WHEN s.is_returned = 0 THEN s.total_revenue ELSE 0 END), 2) AS net_sales_revenue,
    
    -- KPI 2: Profit Calculation
    ROUND(SUM(CASE WHEN s.is_returned = 0 THEN (s.sold_qty * p.profit_per_unit) ELSE 0 END), 2) AS net_profit,
    
    -- KPI 4: Basket Size Logic
    SUM(s.sold_qty) AS total_units_sold,
    COUNT(DISTINCT s.transaction_id) AS total_transactions,
    
    -- KPI 5: Returns
    SUM(s.is_returned) AS return_count,
    
    -- KPI 8: Discounts
    ROUND(SUM(CASE WHEN s.is_returned = 0 THEN s.discount_loss_amount ELSE 0 END), 2) AS total_discount_loss,
    
    -- KPI 9: Customer Tracking
    COUNT(DISTINCT CASE WHEN s.customer_id != 'UNKNOWN_CUSTOMER' THEN s.customer_id END) AS unique_customers

FROM de_mini_project.gold.fact_sales s
JOIN de_mini_project.gold.dim_product p ON s.product_id = p.product_id
GROUP BY 1, 2, 3
""")